# BeltWatch: train on Colab

Runs the data pipeline, the baselines, and the U-Net on a Colab GPU. See `docs/training.md`.

**Runtime → Change runtime type → GPU** first.

**After a disconnect** (usage limit, idle timeout): reconnect and run all cells again from the top. Everything that matters is on Google Drive, so each cell picks up where it stopped: the archive is not downloaded again, a trained forest is reused, and training resumes from the last finished epoch.

## Persist outputs on Google Drive
The archive (7.5 GB), checkpoints, reports, the MLflow database, and a copy of the dataset-version files live on Drive and survive a disconnect. Drive needs about 9 GB free. The extracted images stay on the Colab disk: reading thousands of small files from Drive is slow, and extracting from the Drive copy takes minutes.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/beltwatch-runs"
!mkdir -p {WORK}/downloads {WORK}/dataset-version
!df -h /content/drive | tail -1

## Set up

In [ ]:
import os

if os.path.isdir("/content/beltwatch"):
    # A reused runtime keeps its old clone; update it so every cell runs the latest code.
    !git -C /content/beltwatch pull --ff-only
else:
    !git clone https://github.com/professor3333/beltwatch.git /content/beltwatch
%cd /content/beltwatch
!git log -1 --oneline
# Older versions of this notebook linked data/downloads to Drive; the archive now downloads locally.
if os.path.islink("data/downloads"):
    os.unlink("data/downloads")
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
!uv sync --extra cu126
!uv run python -c "import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

## Build the dataset
The 7.5 GB archive downloads to the Colab disk: streaming it straight into Drive ran at about 1 MB/s and Zenodo dropped the connection. Once verified, it is copied to Drive. Later sessions copy it back from Drive (a few minutes) and only verify and extract it. A download cut off by a disconnect starts over in the next session.

No GPU is needed up to and including this cell, so a CPU runtime saves GPU quota.

In [ ]:
ARCHIVE = "zerowaste-f-final.zip"
DRIVE_ARCHIVE = f"{WORK}/downloads/{ARCHIVE}"
LOCAL_ARCHIVE = f"data/downloads/{ARCHIVE}"
# Copies go through a temporary name so an interrupted copy never looks like a finished archive.
!mkdir -p data/downloads
if os.path.exists(DRIVE_ARCHIVE) and not os.path.exists(LOCAL_ARCHIVE):
    !cp {DRIVE_ARCHIVE} {LOCAL_ARCHIVE}.copying && mv {LOCAL_ARCHIVE}.copying {LOCAL_ARCHIVE}
!uv run dvc repro
# The downloader names the archive only after verifying it, so this copies a verified file.
if os.path.exists(LOCAL_ARCHIVE) and not os.path.exists(DRIVE_ARCHIVE):
    !cp {LOCAL_ARCHIVE} {DRIVE_ARCHIVE}.copying && mv {DRIVE_ARCHIVE}.copying {DRIVE_ARCHIVE}
# Keep the dataset-version record on Drive until it is committed through a PR.
!cp -r data/manifests data/splits dvc.lock {WORK}/dataset-version/
!cat data/splits/zerowaste-f-splits.json | head -40

## Classical baselines (val only)

In [ ]:
RF_DIR = f"{WORK}/models/random-forest"
if os.path.exists(f"{RF_DIR}/model.joblib"):
    print("reusing", RF_DIR)
else:
    !uv run python -m beltwatch.baselines.random_forest --out {RF_DIR}
!uv run python scripts/evaluate.py --model all-background --split val --out-dir {WORK}/reports
!uv run python scripts/evaluate.py --model random-forest --model-dir {RF_DIR} --split val --out-dir {WORK}/reports

## U-Net: write a Drive-backed config, time it, then train

In [ ]:
import yaml


def colab_config(base: str, out: str, output_dir: str, run_name: str | None = None) -> None:
    with open(base) as fh:
        c = yaml.safe_load(fh)
    c["output_dir"] = output_dir
    c["tracking"]["mlflow_uri"] = f"sqlite:///{WORK}/mlflow.db"
    c["runtime"]["num_workers"] = 2
    if run_name:
        c["run_name"] = run_name
    with open(out, "w") as fh:
        yaml.safe_dump(c, fh)


UNET_DIR = f"{WORK}/models/unet-resnet18-ce"
colab_config("configs/unet.yaml", "/content/unet-colab.yaml", UNET_DIR)
# The timing run gets its own directory so re-running it never overwrites the full run's latest.pt.
colab_config(
    "configs/unet.yaml",
    "/content/unet-timing.yaml",
    f"{WORK}/models/unet-timing",
    "unet-resnet18-ce-timing",
)

In [ ]:
# Timing only; skip it once you have the epoch time.
!uv run python scripts/train.py --config /content/unet-timing.yaml --max-steps 300 --max-val-images 50

In [ ]:
# Full run. Resumes automatically from the last finished epoch after a disconnect.
RESUME = f"--resume {UNET_DIR}/latest.pt" if os.path.exists(f"{UNET_DIR}/latest.pt") else ""
print("resume:", RESUME or "no, starting fresh")
!uv run python scripts/train.py --config /content/unet-colab.yaml {RESUME}

In [ ]:
!uv run python scripts/evaluate.py --model unet --model-dir {UNET_DIR} --split val --out-dir {WORK}/reports

## SegFormer-B0 challenger (matched configuration)

In [ ]:
SEG_DIR = f"{WORK}/models/segformer-b0-ce"
colab_config("configs/segformer_b0.yaml", "/content/segformer-colab.yaml", SEG_DIR)

In [ ]:
RESUME = f"--resume {SEG_DIR}/latest.pt" if os.path.exists(f"{SEG_DIR}/latest.pt") else ""
print("resume:", RESUME or "no, starting fresh")
!uv run python scripts/train.py --config /content/segformer-colab.yaml {RESUME}
!uv run python scripts/evaluate.py --model segformer --model-dir {SEG_DIR} --split val --out-dir {WORK}/reports

## Keep the dataset version
Commit `data/manifests/`, `data/splits/` and `dvc.lock` from this run through a PR. A copy sits in `beltwatch-runs/dataset-version/` on Drive, so you can also commit it from your laptop.

In [ ]:
!git status --short data dvc.lock